In [0]:
import dlt
from pyspark.sql.functions import col, sum, avg, countDistinct, year, month

# GOLD LAYER PIPELINE
@dlt.table(
    comment="Aggregated race results with driver, constructor, and race info."
)
def f1_race_results():
    return (
        dlt.read("silver_race_results")
        .groupBy("race_id", "year", "round", "driver_id", "constructor_id")
        .agg(
            sum("points").alias("total_points"),
            avg("points").alias("avg_points"),
            countDistinct("position").alias("positions_covered")
        )
    )

@dlt.table(
    comment="Driver standings over seasons."
)
def f1_driver_standings():
    return (
        dlt.read("f1_race_results")
        .groupBy("year", "driver_id")
        .agg(
            sum("total_points").alias("season_points"),
            avg("avg_points").alias("avg_points_per_race")
        )
        .orderBy(col("season_points").desc())
    )

@dlt.table(
    comment="Constructor standings over seasons."
)
def f1_constructor_standings():
    return (
        dlt.read("f1_race_results")
        .groupBy("year", "constructor_id")
        .agg(
            sum("total_points").alias("constructor_points")
        )
        .orderBy(col("constructor_points").desc())
    )
